<a href="https://colab.research.google.com/github/Ruchirr999/csot-ml-astronomy/blob/main/submissions/Ruchirr999/week2/week2_mlp_starter_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSoT'26 - ML in Astronomy - Week 2 . Part 2: Your First Neural Network (Starter)

**Goal:** Define a 2-layer fully-connected network (MLP) with `nn.Module`, forward-pass a real batch, set up a loss + optimiser, and run a single optimisation step to watch the loss drop.

**Before you begin:**
1. Switch this notebook to a **GPU runtime** (`Runtime -> Change runtime type -> GPU`).
2. Read [`04-building-models-with-nn-module.md`](../04-building-models-with-nn-module.md) and [`05-loss-functions-and-optimisers.md`](../05-loss-functions-and-optimisers.md).

Replace each `TODO` with working code. **Do not** open the solution until you've genuinely attempted every TODO. (We *set up* training here; the full multi-epoch training loop is Week 3.)

## Step 0 — Re-create the Week 1 data pipeline

Week 2 builds directly on the `DataLoader`s from Week 1. The cells below reproduce that pipeline (download is commented out — uncomment it the first time, exactly as in [`week1_data_solution.ipynb`](../../Week-1/notebooks/week1_data_solution.ipynb)). If you saved `galaxy_data/` to Google Drive in Week 1, just re-mount Drive and point `DATA_ROOT` at it instead of re-downloading.

After this section you should have `train_loader`, `val_loader`, `test_loader`, `train_ds`, and `num_classes`.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cuda


In [ ]:
RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2/images"   # adjust if your JPGs landed one folder deeper
DATA_ROOT = Path("galaxy_data")        # we create train/val/test subfolders here
LABELS_URL = "https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz"


# Download Galaxy Zoo 2 images dataset
!kaggle datasets download -d jaimetrickz/galaxy-zoo-2-images
# Create raw data folder
!mkdir -p galaxy_raw
# Unzip dataset
!unzip -q galaxy-zoo-2-images.zip -d galaxy_raw
# Download Hart et al. labels
!wget -q -O galaxy_raw/gz2_hart16.csv.gz \
https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz
# Unzip labels csv
!gunzip -f galaxy_raw/gz2_hart16.csv.gz

print("RAW_ROOT   =", RAW_ROOT)
print("IMAGES_DIR =", IMAGES_DIR)
print("DATA_ROOT  =", DATA_ROOT)

print("RAW_ROOT contents:", sorted(p.name for p in RAW_ROOT.iterdir()))
jpg_count = sum(1 for _ in IMAGES_DIR.glob("*.jpg"))
print(f"Flat JPG count in {IMAGES_DIR}: {jpg_count:,}")

print("\nMapping CSV preview:")
print(pd.read_csv(RAW_ROOT / "gz2_filename_mapping.csv", nrows=3))

print("\nLabels CSV preview (note dr7objid — we rename to objid before merging):")
print(pd.read_csv(RAW_ROOT / "gz2_hart16.csv", nrows=3)[["dr7objid", "gz2_class"]])

def high_level_label(gz2_class: str):
    """Collapse detailed GZ2 codes (Sc2t, Ei, SBb2m, …) to a few training buckets."""
    if not isinstance(gz2_class, str) or gz2_class == "A":
        return None  # artifact / ambiguous
    if gz2_class.startswith("E"):
        return "elliptical"
    if gz2_class.startswith("SB"):
        return "spiral_barred"
    if gz2_class.startswith("S"):
        return "spiral"
    return None
def load_labeled_table(mapping_csv, labels_csv):
    """Join Kaggle mapping (objid ↔ asset_id) with Hart et al. morphology labels."""
    mapping = pd.read_csv(mapping_csv)
    labels = pd.read_csv(labels_csv)
    if "dr7objid" in labels.columns:
        labels = labels.rename(columns={"dr7objid": "objid"})
    df = mapping.merge(labels[["objid", "gz2_class"]], on="objid", how="inner")
    df["label"] = df["gz2_class"].map(high_level_label)
    df = df.dropna(subset=["label"]).reset_index(drop=True)
    return df


def _link_image(src: Path, dst: Path) -> bool:
    """Symlink if possible; otherwise copy (some Drive setups block symlinks)."""
    if dst.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(src.resolve(), dst)
    except OSError:
        import shutil
        shutil.copy2(src, dst)
    return True

def build_split_imagefolder_layout(
    images_dir,
    df,
    out_root,
    per_class=200,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
):
    """Create out_root/{train,val,test}//*.jpg for ImageFolder."""
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6
    images_dir = Path(images_dir)
    out_root = Path(out_root)
    summary = {}

    for label in sorted(df["label"].unique()):
        rows = df[df["label"] == label].sample(frac=1, random_state=seed)
        if len(rows) > per_class:
            rows = rows.head(per_class)

        n = len(rows)
        n_train = int(train_frac * n)
        n_val = int(val_frac * n)
        n_test = n - n_train - n_val
        splits = {
            "train": rows.iloc[:n_train],
            "val": rows.iloc[n_train : n_train + n_val],
            "test": rows.iloc[n_train + n_val :],
        }

        summary[label] = {}
        for split_name, split_rows in splits.items():
            linked = 0
            for _, row in split_rows.iterrows():
                src = images_dir / f"{int(row.asset_id)}.jpg"
                dst = out_root / split_name / label / f"{int(row.asset_id)}.jpg"
                if src.exists() and _link_image(src, dst):
                    linked += 1
            summary[label][split_name] = linked
    return summary

df = load_labeled_table(
    RAW_ROOT / "gz2_filename_mapping.csv",
    RAW_ROOT / "gz2_hart16.csv",
)
print("Joined rows:", len(df))
print("\nLabel counts:")
print(df["label"].value_counts())
print("\nExample rows:")
print(df[["asset_id", "objid", "gz2_class", "label"]].head())
PER_CLASS = 200  # increase once the pipeline works (e.g. 2000)

summary = build_split_imagefolder_layout(
    IMAGES_DIR,
    df,
    DATA_ROOT,
    per_class=PER_CLASS,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
)
print("Linked images per class and split:")
print(pd.DataFrame(summary).fillna(0).astype(int))

for split in ("train", "val", "test"):
    split_dir = DATA_ROOT / split
    classes = sorted(p.name for p in split_dir.iterdir() if p.is_dir()) if split_dir.exists() else []
    n_imgs = sum(1 for _ in split_dir.rglob("*.jpg")) if split_dir.exists() else 0
    print(f"{split:5s}: {n_imgs:4d} images  classes={classes}")
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])
train_ds = ImageFolder(root=DATA_ROOT / "train", transform=transform)
val_ds   = ImageFolder(root=DATA_ROOT / "val",   transform=transform)
test_ds  = ImageFolder(root=DATA_ROOT / "test",  transform=transform)

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    print(f"{name:5s}  n={len(ds):4d}  classes={ds.classes}")

print("class_to_idx:", train_ds.class_to_idx)
image, label = train_ds[0]
print("shape :", image.shape)
print("dtype :", image.dtype)
print("label :", label, "->", train_ds.classes[label])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

images, labels = next(iter(train_loader))
print("train batch images:", images.shape)   # (32, 3, 64, 64)
print("train batch labels:", labels.shape)     # (32,)

num_classes = len(train_ds.classes)

Dataset URL: https://www.kaggle.com/datasets/jaimetrickz/galaxy-zoo-2-images
License(s): Attribution 4.0 International (CC BY 4.0)
100% 3.06G/3.06G [02:27<00:00, 22.3MB/s]

RAW_ROOT   = galaxy_raw
IMAGES_DIR = galaxy_raw/images_gz2/images
DATA_ROOT  = galaxy_data
RAW_ROOT contents: ['gz2_filename_mapping.csv', 'gz2_hart16.csv', 'images_gz2']
Flat JPG count in galaxy_raw/images_gz2/images: 243,434

Mapping CSV preview:
                objid    sample  asset_id
0  587722981736120347  original         1
1  587722981736579107  original         2
2  587722981741363294  original         3

Labels CSV preview (note dr7objid — we rename to objid before merging):
             dr7objid gz2_class
0  587732591714893851      Sc+t
1  588009368545984617      Sb+t
2  587732484359913515        Ei
Joined rows: 239100

Label counts:
label
elliptical       97670
spiral           95849
spiral_barred    45581
Name: count, dtype: int64

Example rows:
   asset_id               objid gz2_class       label
0   

## Step 1 - Define the model

A 2-layer MLP: flatten -> Linear(12288, 128) -> ReLU -> Linear(128, num_classes). The output layer returns **raw logits** (no softmax - `CrossEntropyLoss` adds it). Don't forget `super().__init__()`.

In [ ]:
# TODO: define GalaxyMLP(nn.Module) with:
#   self.flatten = nn.Flatten()
#   self.fc1 = nn.Linear(3*64*64, 128); self.relu = nn.ReLU()
#   self.fc2 = nn.Linear(128, num_classes)
# and a forward(self, x) that runs flatten -> fc1 -> relu -> fc2 and returns logits.
import torch.nn as nn

class GalaxyMLP(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3 * 64 * 64, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

## Step 2 - Instantiate and move to the device

Use the real `num_classes` from your data, and `.to(device)` so the model lives where the batches will.

In [ ]:
model = GalaxyMLP(num_classes=num_classes).to(device)

## Step 3 - Inspect the architecture

`print(model)` shows the layers; counting `.parameters()` shows how many weights there are. Notice that the first `Linear` (12288 x 128) dominates - a direct cost of flattening.

In [ ]:
print(model)
print(sum(p.numel() for p in model.parameters()))

GalaxyMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=12288, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)
1573379


## Step 4 - Forward-pass one real batch

Pull a batch from `train_loader`, move it to the device, and run it through the model. The output should be logits of shape `(batch_size, num_classes)`.

In [ ]:
images, labels = next(iter(train_loader));
images = images.to(device); labels = labels.to(device)
logits = model(images)
print(logits.shape)
#       logits = model(images); print logits.shape and confirm it's (B, num_classes).

torch.Size([32, 3])


## Step 5 - Loss and optimiser

`CrossEntropyLoss` consumes raw logits + integer labels. `Adam` with `lr=1e-3` is the sensible default.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## Step 6 - Sanity-check the starting loss

For an untrained model on `C` balanced classes the loss should sit near `ln(C)`. If it's wildly off (or `nan`), suspect label dtype, a stray softmax, or unnormalised inputs.

In [ ]:
import math
from math import log
loss = criterion(logits, labels);
print ("actual entropy = ", loss.item())
print("expected entropy = ", math.log(num_classes))
print ("diff = ", math.log(num_classes) - loss.item())

actual entropy =  1.1507381200790405
expected entropy =  1.0986122886681098
diff =  -0.052125831410930745


## Step 7 - One optimisation step (learning, in miniature)

The three lines at the heart of every training loop: clear gradients, backprop, update. Re-evaluate the loss on the *same* batch afterwards - it should drop. (Week 3 repeats this over all batches for many epochs.)

In [ ]:
model.train()

optimizer.zero_grad()

logits = model(images)          # fresh forward pass
loss = criterion(logits, labels)

old_loss = loss.item()

loss.backward()
optimizer.step()

new_logits = model(images)
new_loss = criterion(new_logits, labels)

print("old loss:", old_loss)
print("new loss:", new_loss.item())

old loss: 1.1507381200790405
new loss: 7.133986473083496


## Step 8 (stretch) - How big does the model get?

Re-build the MLP with different hidden widths and print the parameter count each time. The first `Linear` (in_features x hidden) dominates - flattening is expensive. A CNN (Week 3) does more with far fewer parameters by sharing weights across the image.

In [ ]:
# TODO (optional): rebuild the MLP with hidden widths 64/128/256/512 and print param counts.

## Reflection *(write 2-3 sentences each)*

1. Your untrained loss should have been near `ln(num_classes)`. Why is that the expected starting value?
2. After one step the loss dropped on that batch. Why is that *not yet* the same as 'the model is trained'?
3. Compare the MLP's parameter count to what you'd expect a CNN to need. Why does flattening make the first layer so large?

*(Replace this prompt with your answers.)*